# KIST PacBio Full-Length 16S · P92 / 92개 중 PC·NC + 이상 3개 제외 (87개)
원본 Excel 기준 **상단 92개 (P92)**와 **하단 18개 (P18)**는 별도 프로젝트이며, 함께 분석하지 않습니다.

**이 노트북은 P92 / no_controls_no_three / 예상 샘플 87개만 분석**합니다.

- P92: KCT / K902 / K905의 W1–W3, PC 및 NC 각 1개 (총 92개)
- P18: KUW / KUD / KCR / KDR 각 4개, PC 및 NC 각 1개 (총 18개)
- P92 제외 대상 3개: K905_W1_10 (FKDN260619042-1A), KCT_W3_10 (FKDN260619082-1A), K902_W3_02 (FKDN260619084-1A).
- PC, NC는 프로젝트마다 별개의 FKDN ID입니다.

**분석**: **PC와 NC는 절대 분석하지 않습니다.** FASTQ 분석 시 manifest 단계부터 제외한 후 각 노트북별 DADA2 CCS → ASV를 생성합니다. 기존 결과 ZIP의 ASV table을 재사용할 경우에는 NC/PC를 ASV 테이블에서 제거하고 다양성 지표를 다시 계산합니다.
세 프로젝트 제외 샘플 3개는 **P92 87개 분석에만** 반영합니다. 결과는 프로젝트별 Google Drive 폴더에 개별 저장합니다.

**입력 주의:** 기존 2.17GB ZIP이 원시 FASTQ를 포함하는지는 미확인입니다. FASTQ와 검증된 primer가 없으면 신규 DADA2 분석은 실행할 수 없습니다. 양쪽 프로젝트 PC/NC 샘플을 구별할 수 없는 ASV table은 분석을 중지합니다.

**2026-11-12 FASTQ-only update:** Execute from FASTQ inside ZIP or Drive/raw_fastq; no ASV table assumed. The analyzed input never includes PC/NC. Provide the verified amplification forward primer before QIIME 2 installation.


In [ ]:
# 0. Project- and scenario-specific configuration
DRIVE_PROJECT_ID = "1070GDz6WnKV23Ygzgs08idwO1p4-xYm5"
DRIVE_METADATA_XLSX_ID = "1_USCzdwWfQ614tjzDocbGDcM5aZp-YBD"
DRIVE_EXISTING_ZIP_ID = "1rPpGDmZHDcjNAm0UmjP1kZl5VdGB9478"
DRIVE_FASTQ_FOLDER_ID = "1ujVQlcJtzMRYEqGone8Dw_viN53IzjyB"
DRIVE_REPORT_FOLDER_ID = "1ftL6PEvdc7QAUmW6oAr762cfRglCFp4i"
DRIVE_REFERENCE_FOLDER_ID = "1Ed4eWpq7kCTaJG0A81PZ42tfX2mtga7-"
GITHUB_REF = "feat/full-length-16s-hifi-pipeline-20261112"
PROJECT = "P92"
RUN_SCENARIO = "no_controls_no_three"
EXPECTED_SAMPLES = 87
DOWNLOAD_VENDOR_ZIP = True  # Read ZIP only when Drive/raw_fastq has no individual FASTQs.
DOWNLOAD_RAW_FASTQ = True  # Fetch only included FASTQs from Drive, never PC/NC.
RUN_NEW_DADA2 = True  # Perform actual FASTQ-to-ASV inference (requires verified primer).
INSTALL_QIIME = True  # Install 2026.10; resource-intensive in Colab.
QIIME_CONDA_ENV = "rachis-qiime2-2026.10"
FORWARD_PRIMER_5TO3 = ""     # Do not invent this; verify from original amplicon protocol.
REVERSE_PRIMER_5TO3 = ""     # Optional; must be checked for correct orientation.
from pathlib import Path
WORK = Path("/content/kist_full_length_16s/P92/no_controls_no_three")
WORK.mkdir(parents=True, exist_ok=True)
print("Running ONLY:",PROJECT,RUN_SCENARIO, "Expected samples:", EXPECTED_SAMPLES)


In [ ]:
# 1. Authenticate Drive and fetch Python code from GitHub
import os, sys, json, io, shutil, subprocess, requests, zipfile
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive", force_remount=False)
subprocess.run([sys.executable,"-m","pip","install","-q","pandas","numpy","scipy","matplotlib","openpyxl"],check=True)
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import google.auth, pandas as pd
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api = build("drive","v3",credentials=creds,cache_discovery=False)
def list_drive(folder):
    files, token = [], None
    while True:
        page = drive_api.files().list(q="'"+folder+"' in parents and trashed = false",
               fields="nextPageToken,files(id,name,size,mimeType)",pageSize=1000,pageToken=token).execute()
        files.extend(page.get("files",[]))
        token=page.get("nextPageToken")
        if not token: break
    return files
def download_from_drive(file_id, target):
    target = Path(target)
    if target.exists() and target.stat().st_size > 0:
        print("Cached:",target.name); return target
    target.parent.mkdir(parents=True,exist_ok=True)
    req=drive_api.files().get_media(fileId=file_id)
    with target.open("wb") as f:
        dl=MediaIoBaseDownload(f,req,chunksize=16*1024*1024)
        done=False
        while not done:
            _,done=dl.next_chunk()
    return target
def github_file(path):
    url="https://raw.githubusercontent.com/kimtk94/Codex/"+GITHUB_REF+"/full_length_16s/"+path
    response=requests.get(url,timeout=60); response.raise_for_status()
    result=WORK/path; result.parent.mkdir(parents=True,exist_ok=True)
    result.write_bytes(response.content)
    return result
MAIN=github_file("src/full16s.py")
SUBSETS=github_file("src/subset_analyses.py")
PROJECT_SCRIPT=github_file("src/project_splits.py")
FASTQ_SCRIPT=github_file("src/fastq_inputs.py")
QIIME_SCRIPT=github_file("scripts/run_qiime2.sh")
print("GitHub files retrieved",MAIN.name,SUBSETS.name,PROJECT_SCRIPT.name,QIIME_SCRIPT.name)

In [ ]:
# 2. Verify original Excel 110 rows, split first 92 and last 18 by row order.
XLSX = download_from_drive(DRIVE_METADATA_XLSX_ID, WORK/"metadata/Pacbio_Analysis_Form.xlsx")
META = WORK/"metadata/sample_metadata_110.tsv"
subprocess.run([sys.executable,str(MAIN),"samples",str(XLSX),str(META)],check=True)
SPLIT_DIR = WORK/"metadata"/"projects"
subprocess.run([sys.executable,str(PROJECT_SCRIPT),"split","--metadata",str(META),
                "--output-dir",str(SPLIT_DIR)],check=True)
SCOPED_META = SPLIT_DIR/(PROJECT+"_metadata.tsv")
metadata = pd.read_csv(SCOPED_META,sep="\t",dtype=str).fillna("")
assert len(metadata)==(92 if PROJECT=="P92" else 18)
assert metadata["control"].isin(["PC","NC"]).sum()==2
assert metadata["control"].value_counts().get("PC",0)==1
assert metadata["control"].value_counts().get("NC",0)==1
print("Project",PROJECT, "reference rows",len(metadata),"PC/NC",metadata["control"].value_counts().to_dict())
# This metadata is for provenance only. The analysis manifest MUST exclude PC/NC.
analysis_metadata = metadata.loc[~metadata["control"].isin(["PC","NC"])].copy()
if PROJECT == "P92" and RUN_SCENARIO == "no_controls_no_three":
    EXCLUDE_FKDN = {"FKDN260619042-1A","FKDN260619082-1A","FKDN260619084-1A"}
    analysis_metadata = analysis_metadata.loc[~analysis_metadata["novogene-id"].isin(EXCLUDE_FKDN)].copy()
ANALYSIS_META = WORK/"metadata"/"analysis_metadata_no_PC_NC.tsv"
assert len(analysis_metadata) == EXPECTED_SAMPLES, f"Expected {EXPECTED_SAMPLES}, got {len(analysis_metadata)}"
assert not analysis_metadata["sample-name"].str.upper().isin(["PC","NC"]).any()
analysis_metadata.to_csv(ANALYSIS_META, sep="\t", index=False)
print("Analysis samples:",len(analysis_metadata),"controls:",analysis_metadata["control"].value_counts().to_dict())
display(analysis_metadata[["excel_order","sample-id","sample-name","novogene-id","cohort","week","control"]])


In [ ]:
# 3. Project-specific control IDs and three validated P92 exclusions
CONTROLS=metadata.loc[metadata["control"].isin(["PC","NC"]),["sample-id","novogene-id","control"]]
print("Control IDs:",CONTROLS.to_dict("records"))
P92_EXCLUSIONS=[
    ("K905_W1_10","FKDN260619042-1A"),
    ("KCT_W3_10","FKDN260619082-1A"),
    ("K902_W3_02","FKDN260619084-1A")
]
if PROJECT == "P92":
    for name,fkdn in P92_EXCLUSIONS:
        x = metadata.loc[metadata["novogene-id"]==fkdn]
        assert len(x)==1 and x["sample-name"].item()==name, "Excluded sample mismatch: "+fkdn
    print("P92 confirmed three exclusions:",P92_EXCLUSIONS)
else:
    assert not any(metadata["novogene-id"].isin([x[1] for x in P92_EXCLUSIONS]))
    print("P18 never excludes the three P92 samples.")
print("Analysis preset:",RUN_SCENARIO,"target",EXPECTED_SAMPLES)


In [ ]:
# 4. FASTQ source discovery (ZIP is optional; only download if needed)
ZIP=WORK/"vendor/Result_X209SC26096091-Z01-F003_16S.zip"
FEATURE=WORK/"qiime2_outputs/feature-table.tsv"
TAXONOMY=WORK/"qiime2_outputs/taxonomy_export/taxonomy.tsv"
RAW_ITEMS=[x for x in list_drive(DRIVE_FASTQ_FOLDER_ID)
           if x["name"].lower().endswith((".fastq.gz",".fq.gz",".fastq",".fq"))]
print("FASTQs available directly in Drive/raw_fastq:",len(RAW_ITEMS))
if RAW_ITEMS:
    print("Using direct sample FASTQs. No ZIP download.")
elif DOWNLOAD_VENDOR_ZIP:
    ZIP=download_from_drive(DRIVE_EXISTING_ZIP_ID,ZIP)
    subprocess.run([sys.executable,str(MAIN),"inspect",str(ZIP),
                    str(WORK/"vendor/zip_inventory.json")],check=True)
    with zipfile.ZipFile(ZIP) as archive:
        fastq_members=[item.filename for item in archive.infolist()
                       if item.filename.lower().endswith((".fastq.gz",".fq.gz",".fastq",".fq"))]
    print("FASTQs inside ZIP:",len(fastq_members))
    if not fastq_members:
        raise FileNotFoundError("No FASTQ inside ZIP. Upload original HiFi FASTQ files to Drive/raw_fastq.")
else:
    raise FileNotFoundError("No FASTQ input. Upload to Drive/raw_fastq or enable DOWNLOAD_VENDOR_ZIP.")

In [ ]:
# 5. Strict project/scenario FASTQ manifest: PC and NC never enter DADA2.
FASTQ_DIR=WORK/"raw_fastq"
PREP_OUT=WORK/"prepared_inputs"
MANIFEST=PREP_OUT/"manifest.tsv"
if RAW_ITEMS:
    sys.path.insert(0,str(FASTQ_SCRIPT.parent))
    from fastq_inputs import classify
    approved=analysis_metadata.to_dict("records")
    selected=[item for item in RAW_ITEMS if classify(item["name"],approved) is not None]
    print("Eligible direct sample FASTQ files:",len(selected))
    if not selected:
        raise ValueError("No filenames match included sample IDs. Use Novogene FKDN ID in FASTQ filename.")
    FASTQ_DIR.mkdir(parents=True,exist_ok=True)
    for item in selected:
        download_from_drive(item["id"],FASTQ_DIR/item["name"])
    cmd=[sys.executable,str(FASTQ_SCRIPT),"--metadata",str(META),
         "--project",PROJECT,"--scenario",RUN_SCENARIO,
         "--output",str(PREP_OUT),"--directory",str(FASTQ_DIR)]
else:
    if not ZIP.exists():
        raise FileNotFoundError("ZIP not found; upload FASTQ to Drive/raw_fastq.")
    cmd=[sys.executable,str(FASTQ_SCRIPT),"--metadata",str(META),
         "--project",PROJECT,"--scenario",RUN_SCENARIO,
         "--output",str(PREP_OUT),"--zip",str(ZIP)]
subprocess.run(cmd,check=True)
counts=pd.read_csv(MANIFEST,sep="\t")
assert len(counts)==EXPECTED_SAMPLES, (len(counts),EXPECTED_SAMPLES)
assert set(counts["sample-id"])==set(analysis_metadata["sample-id"])
assert not counts["sample-id"].str.contains("PC__|NC__",regex=True).any()
subprocess.run([sys.executable,str(MAIN),"qc",str(MANIFEST),
                str(WORK/"metadata/fastq_qc.tsv")],check=True)
print("PASS: FASTQ-only manifest",PROJECT,RUN_SCENARIO,len(counts))

In [ ]:
# 6. (Optional, heavy) install QIIME 2 2026.10 + denoise PacBio HiFi.
# Direct official environment: https://library.qiime2.org/quickstart/qiime2
# Colab may lack resources for the full environment. GitHub Actions only handles CI.
if RUN_NEW_DADA2 and not FORWARD_PRIMER_5TO3:
    raise ValueError("Set the verified 5-prime amplification primer in FORWARD_PRIMER_5TO3.")
if RUN_NEW_DADA2 and not MANIFEST.exists():
    raise FileNotFoundError("FASTQ manifest absent. Run cell 5 first.")
if INSTALL_QIIME or RUN_NEW_DADA2:
    import tarfile
    micromamba=WORK/"bin/micromamba"
    if not micromamba.exists():
        url="https://micro.mamba.pm/api/micromamba/linux-64/latest"
        res=requests.get(url,timeout=180);res.raise_for_status()
        with tarfile.open(fileobj=io.BytesIO(res.content),mode="r:bz2") as tar:
            member=next(item for item in tar.getmembers() if item.name=="bin/micromamba")
            micromamba.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(member) as src,micromamba.open("wb") as dst:shutil.copyfileobj(src,dst)
        micromamba.chmod(0o755)
    env={**os.environ,"MAMBA_ROOT_PREFIX":str(WORK/"micromamba")}
    yaml="https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.10/qiime2/released/rachis-qiime2-linux-64-conda.yml"
    if INSTALL_QIIME:
        subprocess.run([str(micromamba),"create","-y","-n",QIIME_CONDA_ENV,"-f",yaml],check=True,env=env)
    if RUN_NEW_DADA2:
        if not MANIFEST.exists(): raise FileNotFoundError("Upload FASTQs and run cell 5.")
        if not FORWARD_PRIMER_5TO3: raise ValueError("Required actual forward primer not configured.")
        candidate_qza=[x for x in list_drive(DRIVE_REFERENCE_FOLDER_ID) if x["name"].lower().endswith(".qza")]
        CLASSIFIER=""
        if candidate_qza:
            item=candidate_qza[0]
            CLASSIFIER=str(download_from_drive(item["id"],WORK/"references"/item["name"]))
        cmd=[str(micromamba),"run","-n",QIIME_CONDA_ENV,"bash",str(QIIME_SCRIPT),
            str(MANIFEST),str(FEATURE.parent),FORWARD_PRIMER_5TO3,CLASSIFIER,REVERSE_PRIMER_5TO3]
        subprocess.run(cmd,env=env,check=True)
else:
    print("QIIME 2 optional denoising not requested.")

In [ ]:
# 7. Recompute this project's ASV/alpha/beta/report for ONLY the selected preset.
OUTPUT=WORK/"analysis"
if FEATURE.exists():
    cmd=[sys.executable,str(PROJECT_SCRIPT),"run",
         "--feature-table",str(FEATURE),
         "--metadata",str(META),
         "--project",PROJECT,
         "--scenario",RUN_SCENARIO,
         "--output-dir",str(OUTPUT)]
    if TAXONOMY.exists():
        cmd.extend(["--taxonomy",str(TAXONOMY)])
    subprocess.run(cmd,check=True)
    summary=json.loads((OUTPUT/"project_run.json").read_text(encoding="utf-8"))
    assert summary["status"]=="PASS" and summary["actual_samples"]==EXPECTED_SAMPLES
    print("PASS:",PROJECT,RUN_SCENARIO, summary)
    print("HTML report:", OUTPUT/"reports"/RUN_SCENARIO/"report.html")
else:
    raise FileNotFoundError("ASV feature-table.tsv not available. Check vendor ZIP content or upload raw HiFi FASTQ and run DADA2 first.")


In [ ]:
# 8. Upload this project/scenario outputs into its own Drive folder.
from datetime import datetime,timezone
extensions={".html",".png",".tsv",".json",".csv"}
files=[]
for source in [WORK/"metadata",WORK/"vendor",WORK/"analysis",WORK/"qiime2_outputs"]:
    if source.exists():
        files += [f for f in source.rglob("*") if f.is_file() and f.suffix.lower() in extensions]
if not (OUTPUT/"project_run.json").exists():
    raise RuntimeError("Refusing Drive upload: scientific analysis has not passed.")
if files:
    name=PROJECT+"_"+RUN_SCENARIO+"_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    folder=drive_api.files().create(body={"name":name,"mimeType":"application/vnd.google-apps.folder",
                 "parents":[DRIVE_REPORT_FOLDER_ID]},fields="id").execute()
    dest=folder["id"]
    for f in files:
        if f.stat().st_size>50*1024*1024:
            print("Skip large output:",f); continue
        filename=str(f.relative_to(WORK)).replace("/","__")
        mime="image/png" if f.suffix==".png" else ("text/html" if f.suffix==".html" else "text/plain")
        media=MediaFileUpload(str(f),mimetype=mime,resumable=True)
        drive_api.files().create(body={"name":filename,"parents":[dest]},media_body=media,fields="id").execute()
    print("PASS: controls and excluded samples are absent. Project results:", "https://drive.google.com/drive/folders/"+dest)
else:
    raise FileNotFoundError("No output files to upload.")
